# What drives the price of a car?

![](../images/kurt.jpeg)

**OVERVIEW**

In this application, you will explore a dataset from Kaggle. The original dataset contained information on 3 million used cars. The provided dataset contains information on 426K cars to ensure speed of processing.  Your goal is to understand what factors make a car more or less expensive.  As a result of your analysis, you should provide clear recommendations to your client -- a used car dealership -- as to what consumers value in a used car.

### CRISP-DM Framework

<center>
    <img src = ../images/crisp.png width = 50%/>
</center>


To frame the task, throughout our practical applications, we will refer back to a standard process in industry for data projects called CRISP-DM.  This process provides a framework for working through a data problem.  Your first step in this application will be to read through a brief overview of CRISP-DM [here](https://mo-pcco.s3.us-east-1.amazonaws.com/BH-PCMLAI/module_11/readings_starter.zip).  After reading the overview, answer the questions below.

### Business Understanding

From a business perspective, we are tasked with identifying key drivers for used car prices.  In the CRISP-DM overview, we are asked to convert this business framing to a data problem definition.  Using a few sentences, reframe the task as a data task with the appropriate technical vocabulary. 

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.io as pio
import plotly.graph_objects as go
import seaborn as sns
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.model_selection import cross_val_score, GridSearchCV, train_test_split, KFold, cross_validate
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures, OrdinalEncoder
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer, make_column_transformer
from sklearn.feature_selection import SequentialFeatureSelector


import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

sns.set_style('whitegrid')
pio.templates.default = 'plotly_white'

In [2]:
df = pd.read_csv('../data/vehicles_enriched.csv')

In [3]:
df.sample(n=5)

,id,region,price,year,manufacturer,model,condition,cylinders,fuel,odometer,title_status,transmission,VIN,drive,size,type,paint_color,state,transmission_style,base_price
100905,7315883828,lakeland,36990,2017.0,ford,f150 super cab lariat,good,6 cylinders,gas,38094.0,clean,other,1FTFX1EG9HKD14814,4wd,NaN,pickup,white,fl,NaN,31195.0
143835,7314599643,"quad cities, IA/IL",27995,2006.0,chevrolet,corvette,good,8 cylinders,gas,NaN,clean,manual,NaN,rwd,NaN,convertible,black,il,NaN,NaN
20235,7308399808,little rock,78423,2015.0,chevrolet,corvette,NaN,8 cylinders,gas,30200.0,clean,automatic,NaN,rwd,NaN,convertible,NaN,ar,NaN,NaN
300734,7312663807,northern panhandle,14000,2013.0,bmw,328i,NaN,NaN,gas,92965.0,clean,automatic,NaN,NaN,NaN,NaN,NaN,oh,NaN,NaN
316249,7315368523,eugene,676,2019.0,chevrolet,suburban ls,NaN,8 cylinders,other,47105.0,clean,automatic,1GNSKGKC7KR124145,NaN,NaN,NaN,black,or,NaN,55095.0


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 426880 entries, 0 to 426879
Data columns (total 20 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   id                  426880 non-null  int64  
 1   region              426880 non-null  object 
 2   price               426880 non-null  int64  
 3   year                426798 non-null  float64
 4   manufacturer        416565 non-null  object 
 5   model               423192 non-null  object 
 6   condition           252776 non-null  object 
 7   cylinders           249202 non-null  object 
 8   fuel                423867 non-null  object 
 9   odometer            422480 non-null  float64
 10  title_status        418638 non-null  object 
 11  transmission        424324 non-null  object 
 12  VIN                 265838 non-null  object 
 13  drive               296313 non-null  object 
 14  size                120519 non-null  object 
 15  type                334022 non-nul

## Data Dictionary

| Column Name | Description | Data Type | Non Null count |
| ----------- | ----------- | --------- | -------------- |
| id | id of listing | int64 | 426880 |
| region | region listed in | object | 426880 |
| price | price of car (our model should aim to predict this) | int64 | 426880 |
| year | year the car was manufactured | int64 | 426880 |
| manufacturer | name of the manufacturer | string | 416565 |
| model | name of the model | string | 423192 |
| condition | condition of the car | string | 252776 |
| cylinders | number of cylinders | string | 249202 |
| fuel | type of fuel used | string | 423867 |
| odometer | odometer reading | float64 | 422480 |
| title_status | title status of the car | string | 418638 |
| transmission | type of transmission | string | 424324 |
| VIN | Vehicle Identification Number - globally unique id | string | 265838 |
| drive | type of drive | string | 296313 |
| size | size of the car | string | 120519 |
| type | type of the car | string | 334022 |
| paint_color | color of the car | string | 296677 |
| state | state where the car is located | string | 426880 |
| transmission_style | style of the transmission (enriched) | string | 77851 |
| base_price | base purchase price of the car (enriched) | float64 | 50869 |


In [5]:
# exploring the data in data wrangler, VIN column has duplicates
# so let us remove the duplicates so that one vehicle listed in multiple regions does not over power our final model
# however, we will keep the rows with missing VINs as they may contain useful information and we don't want to lose them (yet)

df_deduped = pd.concat([
    df[df['VIN'].isna()],
    df[df['VIN'].notna()].drop_duplicates(subset='VIN', keep='first')
])

df_deduped.info()



<class 'pandas.core.frame.DataFrame'>
Index: 279288 entries, 0 to 426833
Data columns (total 20 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   id                  279288 non-null  int64  
 1   region              279288 non-null  object 
 2   price               279288 non-null  int64  
 3   year                279209 non-null  float64
 4   manufacturer        269131 non-null  object 
 5   model               275620 non-null  object 
 6   condition           162423 non-null  object 
 7   cylinders           173487 non-null  object 
 8   fuel                277519 non-null  object 
 9   odometer            275994 non-null  float64
 10  title_status        275517 non-null  object 
 11  transmission        277954 non-null  object 
 12  VIN                 118246 non-null  object 
 13  drive               193264 non-null  object 
 14  size                98376 non-null   object 
 15  type                198082 non-null  ob

279288 rows after deduplication, down from 426833 rows. We have lost about 147545 rows (35%), but we have removed the duplicates that could bias our model. 
We will proceed with this deduplicated dataset for our analysis and modeling.

Next step is to remove nulls but before we remove for nulls, let us try to fill values where possible. 

Some of the attributes of a car are unique by manufacturer / make and model year. These attributes can be extrapolated and assumed to be the mode of the other values when missing. To be absolutely sure, only fill the values when rest of the values are same for that column when grouped by manufacturer, model and year. 

Initially filled by mode but then realized when examining data that a Chevrolet Corvette from the same model year had a convertible and coupe in the Type column, so introduced a flag to only fill by 100% match. 

Note - cell below runs for 2 mins (be patient)

In [6]:
df_clean = df_deduped.copy()
df_clean.fillna({'condition':'Unknown'}, inplace=True)

def fill_missing_by_group(df, target_cols, group_cols, unanimous_only=False):
    for col in target_cols:
        missing_before = df[col].isna().sum()

        stats = df.groupby(group_cols)[col].agg(
            mode=lambda x: x.mode().iloc[0] if not x.dropna().empty else None,
            diff_count=lambda x: (x.dropna() != x.mode().iloc[0]).sum() if not x.dropna().empty else 0,
            diff_pct=lambda x: (x.dropna() != x.mode().iloc[0]).sum() / len(x.dropna()) * 100 if not x.dropna().empty else 0
        )

        # show groups where values disagree with the mode
        disagreements = stats[stats['diff_count'] > 0].sort_values('diff_pct', ascending=False)
        if not disagreements.empty:
            print(f"\n{col}: groups with values differing from mode ({len(disagreements)} groups):")
            print(disagreements.head(20).to_string())
        else:
            print(f"\n{col}: all groups are unanimous")

        # fill missing values — restrict to unanimous groups if flag is set
        mode_map = stats['mode']
        if unanimous_only:
            mode_map = mode_map[stats['diff_count'] == 0]

        mask = df[col].isna()
        keys = df.loc[mask, group_cols]
        filled = keys.set_index(group_cols).index.map(mode_map)
        df.loc[mask, col] = filled.values
        missing_after = df[col].isna().sum()
        print(f"{col}: {missing_before} values missing -> {missing_after} remaining :: ({missing_before - missing_after} filled)")
    return df

target_cols = ['cylinders', 'fuel', 'transmission', 'size', 'drive', 'base_price']
group_cols = ['manufacturer', 'model', 'year']

df_clean = fill_missing_by_group(df_clean, target_cols, group_cols, unanimous_only=True)

filled_per_col = df_deduped.isna() & df_clean.notna()
print(filled_per_col.sum())
print(f"\nTotal values filled: {filled_per_col.sum().sum():,}")


cylinders: groups with values differing from mode (4096 groups):
                                                    mode  diff_count   diff_pct
manufacturer  model                 year                                       
chevrolet     venture van           2004.0   3 cylinders           2  66.666667
ford          thunderbird           1983.0   4 cylinders           2  66.666667
mercedes-benz c300                  2017.0   4 cylinders           2  66.666667
ram           2500                  1995.0  10 cylinders           6  66.666667
ford          bronco                1988.0   4 cylinders           2  66.666667
saturn        l300                  2001.0   4 cylinders           2  66.666667
mercedes-benz benz ml500            2006.0   4 cylinders           2  66.666667
ford          f150                  2020.0   5 cylinders           2  66.666667
chevrolet     colorado extended cab 2017.0   4 cylinders           2  66.666667
ford          ranger                1984.0   4 cylinde

In [7]:
## review some data after filling nulls
df_clean.loc[(df_clean.manufacturer == 'chevrolet') & (df_clean.model == 'corvette') & (df_clean.year == 2014)]

,id,region,price,year,manufacturer,model,condition,cylinders,fuel,odometer,title_status,transmission,VIN,drive,size,type,paint_color,state,transmission_style,base_price
20596,7304834785,little rock,45000,2014.0,chevrolet,corvette,excellent,8 cylinders,gas,39500.0,clean,automatic,NaN,rwd,NaN,coupe,red,ar,NaN,NaN
53436,7315796340,sacramento,49900,2014.0,chevrolet,corvette,like new,8 cylinders,gas,22560.0,clean,manual,NaN,rwd,NaN,convertible,red,ca,NaN,NaN
77455,7315592079,denver,47599,2014.0,chevrolet,corvette,excellent,8 cylinders,gas,24000.0,clean,manual,NaN,rwd,NaN,NaN,blue,co,NaN,NaN
195414,7314444647,central michigan,36500,2014.0,chevrolet,corvette,excellent,8 cylinders,gas,30000.0,rebuilt,automatic,NaN,rwd,NaN,coupe,grey,mi,NaN,NaN
60706,7316855578,SF bay area,54988,2014.0,chevrolet,corvette,like new,8 cylinders,gas,12390.0,clean,automatic,1G1YD3D79E5115683,rwd,NaN,convertible,black,ca,Automatic,NaN
78036,7315252819,denver,37500,2014.0,chevrolet,corvette,excellent,8 cylinders,gas,39372.0,clean,manual,1G1YJ2D79E5100166,rwd,NaN,coupe,silver,co,Manual/Standard,NaN
111457,7316688821,south florida,50000,2014.0,chevrolet,corvette,like new,8 cylinders,gas,33243.0,clean,automatic,1G1YK2D71E5113457,rwd,compact,hatchback,white,fl,Automatic,NaN
116113,7307157133,tallahassee,48800,2014.0,chevrolet,corvette,excellent,8 cylinders,gas,16307.0,clean,automatic,1G1YF2D73E5100181,rwd,NaN,NaN,yellow,fl,Automatic,NaN
183113,7307327485,baltimore,0,2014.0,chevrolet,corvette,Unknown,8 cylinders,gas,21303.0,clean,manual,1G1YJ2D76E5132279,rwd,NaN,coupe,red,md,Manual/Standard,NaN
189157,7314796691,south coast,48000,2014.0,chevrolet,corvette,like new,8 cylinders,gas,24000.0,clean,automatic,1G1YM3D74E5109337,rwd,mid-size,convertible,red,ma,Automatic,NaN


In [8]:
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 279288 entries, 0 to 426833
Data columns (total 20 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   id                  279288 non-null  int64  
 1   region              279288 non-null  object 
 2   price               279288 non-null  int64  
 3   year                279209 non-null  float64
 4   manufacturer        269131 non-null  object 
 5   model               275620 non-null  object 
 6   condition           279288 non-null  object 
 7   cylinders           212834 non-null  object 
 8   fuel                277802 non-null  object 
 9   odometer            275994 non-null  float64
 10  title_status        275517 non-null  object 
 11  transmission        278364 non-null  object 
 12  VIN                 118246 non-null  object 
 13  drive               219164 non-null  object 
 14  size                135510 non-null  object 
 15  type                198082 non-null  ob

Now, let us drop the columns that we know will not influence the price like the VIN, and the change the index of the df to be the id. 
VIN, size, type, paint_color, transmission_style and base_price columns have the most nulls. 

In [9]:
df_clean.isna().sum()

id                         0
region                     0
price                      0
year                      79
manufacturer           10157
model                   3668
condition                  0
cylinders              66454
fuel                    1486
odometer                3294
title_status            3771
transmission             924
VIN                   161042
drive                  60124
size                  143778
type                   81206
paint_color            91925
state                      0
transmission_style    246857
base_price            255508
dtype: int64

In [10]:
df_clean.drop(columns=['VIN', 'paint_color', 'base_price', 'transmission_style', 'size', 'type'], inplace=True, errors='ignore')
df_clean.dropna(inplace=True)
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 177457 entries, 31 to 426833
Data columns (total 14 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   id            177457 non-null  int64  
 1   region        177457 non-null  object 
 2   price         177457 non-null  int64  
 3   year          177457 non-null  float64
 4   manufacturer  177457 non-null  object 
 5   model         177457 non-null  object 
 6   condition     177457 non-null  object 
 7   cylinders     177457 non-null  object 
 8   fuel          177457 non-null  object 
 9   odometer      177457 non-null  float64
 10  title_status  177457 non-null  object 
 11  transmission  177457 non-null  object 
 12  drive         177457 non-null  object 
 13  state         177457 non-null  object 
dtypes: float64(2), int64(2), object(10)
memory usage: 20.3+ MB


In [11]:

firstqr = df_clean['price'].quantile(0.25)
thirdqr = df_clean['price'].quantile(0.75)
iqr = thirdqr - firstqr
lower_bound = firstqr - 1.5 * iqr
if(lower_bound < 0):
    lower_bound = 0
upper_bound = thirdqr + 1.5 * iqr
outliers = df_clean[(df_clean['price'] < lower_bound) | (df_clean['price'] > upper_bound)]
print(f"Outliers detected: {len(outliers)} ({len(outliers) / len(df_clean) * 100:.2f}%)")

df_clean_minus_outliers = df_clean[(df_clean['price'] > lower_bound) & (df_clean['price'] < upper_bound)]
df_clean_minus_outliers = df_clean_minus_outliers[df_clean_minus_outliers['year'] > 1990]

Outliers detected: 8024 (4.52%)


In [12]:
df_clean_minus_outliers.set_index('id', inplace=True)

In [13]:
df_clean_minus_outliers.describe()

,price,year,odometer
count,151436.000000,151436.000000,1.514360e+05
mean,12871.403207,2010.475310,1.155896e+05
std,9463.937050,5.914418,1.400845e+05
min,1.000000,1991.000000,0.000000e+00
25%,5500.000000,2007.000000,6.785400e+04
50%,9999.000000,2011.000000,1.098890e+05
75%,17999.000000,2015.000000,1.520000e+05
max,40000.000000,2022.000000,1.000000e+07


### Data Understanding

After considering the business understanding, we want to get familiar with our data.  Write down some steps that you would take to get to know the dataset and identify any quality issues within.  Take time to get to know the dataset and explore what information it contains and how this could be used to inform your business understanding.

In [14]:
df_sample = df_clean_minus_outliers.sample(5000, random_state=42)
fig = px.histogram(df_sample, x='price', nbins=50, title='Price Distribution')
fig.write_html('../images/price_distribution.html')
fig.show()

In [15]:
df_sample = df_clean_minus_outliers.sample(5000, random_state=42)
fig = px.scatter(df_sample, x='odometer', y='price', color='condition', hover_data=['manufacturer', 'model', 'year'])
fig.write_html('../images/odometer_vs_price.html')
fig.show()

In [16]:
df_sample = df_clean_minus_outliers.sample(5000, random_state=42)
fig = px.scatter(df_sample, x='year', y='price', color='manufacturer', hover_data=['manufacturer'])
fig.write_html('../images/year_vs_price.html')
fig.show()

In [17]:
df_sample = df_clean_minus_outliers.sample(5000, random_state=42)
fig = px.scatter(df_sample, x='condition', y='price', color='condition', hover_data=['manufacturer', 'model', 'year'])
fig.write_html('../images/condition_vs_price.html')
fig.show()

In [18]:
df_sample = df_clean_minus_outliers.sample(5000, random_state=42)
fig = px.scatter(df_sample, x='price', y='title_status', color='condition', hover_data=['manufacturer', 'model', 'year'])
fig.write_html('../images/price_vs_title_status.html')
fig.show()

In [19]:
# visualize price by state for later analysis of regional price differences and potential inclusion of state as a feature in the model
avg_price_by_state = df_clean_minus_outliers.groupby('state')['price'].mean().reset_index()
avg_price_by_state['state'] = avg_price_by_state['state'].str.upper()
fig = px.choropleth(avg_price_by_state, locations='state', locationmode='USA-states', color='price', scope='usa', title='Average Price by State', color_continuous_scale="Blues", labels={'price':'Average Price'})
fig.write_html('../images/avg_price_by_state.html')
fig.show()

### Data Preparation

After our initial exploration and fine-tuning of the business understanding, it is time to construct our final dataset prior to modeling.  Here, we want to make sure to handle any integrity issues and cleaning, the engineering of new features, any transformations that we believe should happen (scaling, logarithms, normalization, etc.), and general preparation for modeling with `sklearn`. 

In [20]:
df_clean_minus_outliers.info()
df_clean_minus_outliers.nunique()
df_clean_minus_outliers.drop(columns=['region', 'model'], inplace=True)

<class 'pandas.core.frame.DataFrame'>
Index: 151436 entries, 7316356412 to 7302338378
Data columns (total 13 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   region        151436 non-null  object 
 1   price         151436 non-null  int64  
 2   year          151436 non-null  float64
 3   manufacturer  151436 non-null  object 
 4   model         151436 non-null  object 
 5   condition     151436 non-null  object 
 6   cylinders     151436 non-null  object 
 7   fuel          151436 non-null  object 
 8   odometer      151436 non-null  float64
 9   title_status  151436 non-null  object 
 10  transmission  151436 non-null  object 
 11  drive         151436 non-null  object 
 12  state         151436 non-null  object 
dtypes: float64(2), int64(1), object(10)
memory usage: 16.2+ MB


In [21]:
df_clean_minus_outliers.nunique()

price            9868
year               32
manufacturer      108
condition           7
cylinders           8
fuel                5
odometer        67749
title_status        6
transmission        3
drive               3
state              51
dtype: int64

In [29]:
df_final = df_clean_minus_outliers.copy()
df_target = df_final[['price']]
df_final.drop(columns=['price'], inplace=True)  # drop what our models will try to predict

df_final_encoded = pd.get_dummies(df_final, columns=['manufacturer', 'year','condition', 'cylinders', 'transmission', 'fuel', 'drive', 'title_status', 'state'], drop_first=True)


### Modeling

With your (almost?) final dataset in hand, it is now time to build some models.  Here, you should build a number of different regression models with the price as the target.  In building your models, you should explore different parameters and be sure to cross-validate your findings.

In [30]:
## first attempt at linear regression with all features and no regularization

X = df_final_encoded
y = df_target['price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
lin_reg = LinearRegression(fit_intercept=False)
lin_reg.fit(X_train, y_train)
y_pred = lin_reg.predict(X_test)


In [31]:
mse = mean_squared_error(y_test, y_pred)
mae = np.mean(np.abs(y_test - y_pred))
me = np.sqrt(mse)
print(f"Linear Regression MSE: {mse:.2f}")
print(f"Linear Regression RMSE: {me:.2f}")
print(f"Linear Regression MAE: {mae:.2f}")
fig = px.scatter(x=y_test, y=y_pred, title='Actual vs Predicted Prices', labels={'x':'Actual Price', 'y':'Predicted Price'})
fig.write_html('../images/lr_actual_vs_predicted.html')
fig.show()

Linear Regression MSE: 33923216.26
Linear Regression RMSE: 5824.36
Linear Regression MAE: 4071.04


#### With a simple Linear Regression model, there is a to 10 to 15% error (MAE ~4K USD and RMSE ~6K USD) using all available features given our price range was from 0 to 40K USD. 
#### Examining the plot shows negative predicted values. Try with log values.

In [34]:
df_target['log_price'] = np.log(df_target['price'])
y_log = df_target['log_price']

X_train, X_test, y_train_log, y_test_log = train_test_split(X, y_log, test_size=0.3, random_state=42)

pipeline = make_pipeline(LinearRegression(fit_intercept=False))
pipeline.fit(X_train, y_train_log)
y_pred_log = pipeline.predict(X_test)

# lin_reg_log = LinearRegression(fit_intercept=False)
# lin_reg_log.fit(X_train, y_train)
# y_pred_log = lin_reg_log.predict(X_test)

# expo back the log predictions to get them back to the original price scale for evaluation
y_pred_actual = np.expm1(y_pred_log)
y_test_actual = np.expm1(y_test_log)

mse = mean_squared_error(y_test_actual, y_pred_actual)
mae = np.mean(np.abs(y_test_actual - y_pred_actual))
rmse = np.sqrt(mse)
print(f"Linear Regression (log price) MSE: {mse:.2f}")
print(f"Linear Regression (log price) RMSE: {rmse:.2f}")
print(f"Linear Regression (log price) MAE: {mae:.2f}")

fig = px.scatter(x=y_test_actual, y=y_pred_actual, title='Actual vs Predicted Prices (log price model)', labels={'x':'Actual Price', 'y':'Predicted Price'})
fig.add_shape(type='line', x0=y_test_actual.min(), y0=y_test_actual.min(), x1=y_test_actual.max(), y1=y_test_actual.max(), line=dict(color='red', dash='dash'))
fig.write_html('../images/lr_log_actual_vs_predicted.html')
fig.show()

Linear Regression (log price) MSE: 41450428.56
Linear Regression (log price) RMSE: 6438.20
Linear Regression (log price) MAE: 4334.27


In [35]:
col_trfx = make_column_transformer(
    (OneHotEncoder(drop='first', min_frequency=2,handle_unknown='infrequent_if_exist'), 
     ['manufacturer', 'cylinders', 'transmission', 'fuel', 'drive', 'state', 'condition', 'title_status']),
    #(OrdinalEncoder(categories=[['salvage', 'Unknown', 'fair', 'good', 'excellent', 'new', 'like new']]), ['condition']),
    #(OrdinalEncoder(categories=[['salvage', 'parts only', 'missing', 'lien', 'rebuilt', 'clean']]), ['title_status']),
    remainder='passthrough')

lin_reg_pipeline = Pipeline(steps=[
    ('preprocessor', col_trfx),
    ('model', LinearRegression(fit_intercept=False))
])

X_train, X_test, y_train_log, y_test_log = train_test_split(df_final, df_target['log_price'], test_size=0.3, random_state=42)

lin_reg_pipeline.fit(X_train, y_train_log)
y_pred_log_pipeline = lin_reg_pipeline.predict(X_test)

y_test_exp_pipeline = np.expm1(y_test_log)
y_pred_exp_pipeline = np.expm1(y_pred_log_pipeline)
mse_pipline_lin_reg = mean_squared_error(y_test_exp_pipeline, y_pred_exp_pipeline)
mae_pipline_lin_reg = np.mean(np.abs(y_test_exp_pipeline - y_pred_exp_pipeline))
rmse_pipline_lin_reg = np.sqrt(mse_pipline_lin_reg)

print(f"Linear Regression Pipeline (log price) MSE: {mse_pipline_lin_reg:.2f}")
print(f"Linear Regression Pipeline (log price) RMSE: {rmse_pipline_lin_reg:.2f}")
print(f"Linear Regression Pipeline (log price) MAE: {mae_pipline_lin_reg:.2f}")


Linear Regression Pipeline (log price) MSE: 96069073.30
Linear Regression Pipeline (log price) RMSE: 9801.48
Linear Regression Pipeline (log price) MAE: 7044.39


In [36]:
poly_ordinal_ohe = make_column_transformer(
     (OneHotEncoder(drop='first', min_frequency=2,handle_unknown='infrequent_if_exist'), 
     ['manufacturer', 'cylinders', 'transmission', 'fuel', 'drive', 'state', 'condition', 'title_status']),
     (PolynomialFeatures(degree=2, include_bias=False), ['odometer', 'year']),
)

lin_reg_pipeline_poly = Pipeline(steps=[
    ('preprocessor', poly_ordinal_ohe),
    ('model', LinearRegression(fit_intercept=False))
])
lin_reg_pipeline_poly.fit(X_train, y_train_log)
y_pred_log_pipeline_poly = lin_reg_pipeline_poly.predict(X_test)
y_test_exp_pipeline_poly = np.expm1(y_test_log)
y_pred_exp_pipeline_poly = np.expm1(y_pred_log_pipeline_poly)
mse_pipline_lin_reg_poly = mean_squared_error(y_test_exp_pipeline_poly, y_pred_exp_pipeline_poly)
mae_pipline_lin_reg_poly = np.mean(np.abs(y_test_exp_pipeline_poly - y_pred_exp_pipeline_poly))
rmse_pipline_lin_reg_poly = np.sqrt(mse_pipline_lin_reg_poly)
print(f"Linear Regression Pipeline with Polynomial Features (log price) MSE: {mse_pipline_lin_reg_poly:.2f}")
print(f"Linear Regression Pipeline with Polynomial Features (log price) RMSE: {rmse_pipline_lin_reg_poly:.2f}")
print(f"Linear Regression Pipeline with Polynomial Features (log price) MAE: {mae_pipline_lin_reg_poly:.2f}")


Linear Regression Pipeline with Polynomial Features (log price) MSE: 83082079.52
Linear Regression Pipeline with Polynomial Features (log price) RMSE: 9114.94
Linear Regression Pipeline with Polynomial Features (log price) MAE: 6448.35


#### Polynomial features seem to be overfitting the data, as the performance metrics are worse than the linear regression pipeline without polynomial features. The MSE, RMSE, and MAE all increased when polynomial features were added, indicating that the model is not generalizing well to the test data. This suggests that the additional complexity introduced by the polynomial features may be causing the model to fit noise in the training data rather than capturing the underlying patterns that generalize to new data.

In [37]:
# we can see that the polynomial features did not improve our model performance, so we will stick with the simpler linear regression pipeline for now and 
# explore regularization 

# feature selection does not make sense as there are only 2 numerical features and the rest are categorical, but we can still try it out for demonstration purposes 
# and to see which features are most important in our linear regression model

n_features_to_select = 50

numerical_cols = ['year', 'odometer']
categorical_cols = ['manufacturer', 'cylinders', 'transmission', 'fuel', 'drive', 'state', 'condition', 'title_status']

preprocessor = ColumnTransformer(transformers=[
    ('numerical', StandardScaler(), numerical_cols),
    ('categorical', OneHotEncoder(min_frequency=2,handle_unknown='infrequent_if_exist', drop='first', sparse_output=False), categorical_cols)
]).set_output(transform='pandas')

lr_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('selector', SequentialFeatureSelector(Ridge(), n_features_to_select=n_features_to_select)),
    ('model', Ridge())
])

X_train, X_test, y_train_log, y_test_log = train_test_split(df_final, df_target['log_price'], test_size=0.3, random_state=42)

lr_pipeline.fit(X_train, y_train_log)
score = lr_pipeline.score(X_test, y_test_log)
print(f"Pipeline score: {score}")
print(f"Selected features: {lr_pipeline.named_steps['selector'].get_feature_names_out()}")

Pipeline score: 0.30217322225623244
Selected features: ['numerical__year' 'numerical__odometer'
 'categorical__manufacturer_HUMMER' 'categorical__manufacturer_audi'
 'categorical__manufacturer_chrysler' 'categorical__manufacturer_dodge'
 'categorical__manufacturer_honda' 'categorical__manufacturer_jeep'
 'categorical__manufacturer_lexus' 'categorical__manufacturer_mercury'
 'categorical__manufacturer_nissan' 'categorical__manufacturer_porsche'
 'categorical__manufacturer_ram' 'categorical__manufacturer_toyota'
 'categorical__cylinders_12 cylinders'
 'categorical__cylinders_6 cylinders' 'categorical__cylinders_8 cylinders'
 'categorical__transmission_manual' 'categorical__transmission_other'
 'categorical__fuel_electric' 'categorical__fuel_gas'
 'categorical__fuel_hybrid' 'categorical__fuel_other'
 'categorical__drive_fwd' 'categorical__drive_rwd' 'categorical__state_al'
 'categorical__state_ar' 'categorical__state_az' 'categorical__state_ca'
 'categorical__state_co' 'categorical__state

In [38]:
## above step takes 13 mins to run on CPU, so we will not run it for all features, but we can see that the most important features are the year, odometer, a
# nd some of the manufacturer and state dummies.
y_pred_log = lr_pipeline.predict(X_test)

y_pred_actual = np.expm1(y_pred_log)
y_test_actual = np.expm1(y_test_log)

mse  = mean_squared_error(y_test_actual, y_pred_actual)
rmse = np.sqrt(mse)
mae  = np.mean(np.abs(y_test_actual - y_pred_actual))

print(f"Ridge Regression Pipeline with Feature Selection (log price) MSE: {mse:.2f}")
print(f"Ridge Regression Pipeline with Feature Selection (log price) RMSE: {rmse:.2f}")
print(f"Ridge Regression Pipeline with Feature Selection (log price) MAE: {mae:.2f}")

fig = px.scatter(
    x=y_test_actual,
    y=y_pred_actual,
    labels={'x': 'Actual Price', 'y': 'Predicted Price'},
    title='Actual vs Predicted Prices (Ridge + SFS)',
    opacity=0.5
)

# Perfect prediction line
fig.add_shape(
    type='line',
    x0=y_test_actual.min(), y0=y_test_actual.min(),
    x1=y_test_actual.max(), y1=y_test_actual.max(),
    line=dict(color='red', dash='dash')
)
fig.write_html('../images/ridge_sfs_actual_vs_predicted.html')
fig.show()

Ridge Regression Pipeline with Feature Selection (log price) MSE: 44427971.02
Ridge Regression Pipeline with Feature Selection (log price) RMSE: 6665.43
Ridge Regression Pipeline with Feature Selection (log price) MAE: 4549.85


In [39]:

numerical_cols = ['year', 'odometer']
categorical_cols = ['manufacturer', 'cylinders', 'transmission', 'fuel', 'drive', 'state', 'condition', 'title_status']

preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numerical_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore', drop='first', sparse_output=False), categorical_cols)
]).set_output(transform='pandas')

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('sfs', SequentialFeatureSelector(
        Lasso(alpha=0.01),      
        n_features_to_select=50,
        direction='forward',
        scoring='r2',
        cv=5
    )),
    ('model', Lasso(alpha=0.01))
]).set_output(transform='pandas')

pipeline.fit(X_train, y_train_log)


y_pred_log    = pipeline.predict(X_test)
y_pred_actual = np.expm1(y_pred_log)
y_test_actual = np.expm1(y_test_log)

# Errors 
mse  = mean_squared_error(y_test_actual, y_pred_actual)
rmse = np.sqrt(mse)
mae  = np.mean(np.abs(y_test_actual - y_pred_actual))
r2   = pipeline.score(X_test, y_test_log)

#print(f"R²   : {r2:.4f}")
print(f"MAE with Lasso Regression: ${mae:,.2f}")
print(f"RMSE with Lasso Regression: ${rmse:,.2f}")
print(f"MSE  with Lasso Regression: ${mse:,.2f}")

feature_names = pipeline.named_steps['preprocessor'].get_feature_names_out()
selected      = feature_names[pipeline.named_steps['sfs'].get_support()]
print(f"\nSelected features:\n{selected}")

fig = px.scatter(
    x=y_test_actual, y=y_pred_actual,
    labels={'x': 'Actual Price', 'y': 'Predicted Price'},
    title='Actual vs Predicted Prices (Lasso)',
    opacity=0.5
)
fig.add_shape(
    type='line',
    x0=y_test_actual.min(), y0=y_test_actual.min(),
    x1=y_test_actual.max(), y1=y_test_actual.max(),
    line=dict(color='red', dash='dash')
)
fig.write_html('../images/lasso_sfs_actual_vs_predicted.html')
fig.show()

MAE with Lasso Regression: $4,921.16
RMSE with Lasso Regression: $7,134.38
MSE  with Lasso Regression: $50,899,391.93

Selected features:
['num__year' 'num__odometer' 'cat__manufacturer_ALFA ROMEO'
 'cat__manufacturer_AM GENERAL' 'cat__manufacturer_AUDI'
 'cat__manufacturer_BENTLEY' 'cat__manufacturer_BLUE BIRD'
 'cat__manufacturer_BMW' 'cat__manufacturer_BUICK'
 'cat__manufacturer_CADILLAC' 'cat__manufacturer_CHEVROLET'
 'cat__manufacturer_CHRYSLER' 'cat__manufacturer_CODA'
 'cat__manufacturer_DODGE' 'cat__manufacturer_FIAT'
 'cat__manufacturer_FORD' 'cat__manufacturer_FREIGHTLINER'
 'cat__manufacturer_GEM' 'cat__manufacturer_GENESIS'
 'cat__manufacturer_GEO' 'cat__manufacturer_GMC'
 'cat__manufacturer_GREENGO TEK' 'cat__manufacturer_HINO'
 'cat__manufacturer_HONDA' 'cat__manufacturer_HUMMER'
 'cat__manufacturer_HYUNDAI' 'cat__manufacturer_IC BUS'
 'cat__manufacturer_INDIAN MOTORCYCLE' 'cat__manufacturer_INFINITI'
 'cat__manufacturer_INTERNATIONAL' 'cat__manufacturer_ISUZU'
 'cat__man

### Evaluation

With some modeling accomplished, we aim to reflect on what we identify as a high-quality model and what we are able to learn from this.  We should review our business objective and explore how well we can provide meaningful insight into drivers of used car prices.  Your goal now is to distill your findings and determine whether the earlier phases need revisitation and adjustment or if you have information of value to bring back to your client.

##### Ridge Regression with regularization using standard scaler and selecting top 50 features provided comparable results to Linear Regression model. The dataset only has two numerical features and a lot of categorical features. Using OneHotEncoder for some of these columns resulted in about 215 columns. 

##### With the Ridge Regression model, we can identify the top 50 features that are most influential in predicting used car prices. These features include:
1. year
2. odometer
3. manufacturer
4. cylinders
5. transmission
6. fuel type
7. drive type
8. state
9. condition
10. title status

#### With Lasso Regression, we can identify the top features that are most influential in predicting car prices. There is lesser emphasis on manufacturer in Lasso. 
1. year
2. odomoeter
3. manufacturer
4. cylinders
5. fuel type
6. drive type
7. state
8. condition

We see that Lasso has removed transmission and title status as influencing the target price. 

LinearRegression model was faster to compute with comparable results but we need to make sure it is not overfitting and memorizing data. 


#### Model Experiment Results Summary

| # | Model | Target | Encoding | Scaling | Feature Selection | Regularization | MSE | RMSE | MAE |
|---|-------|--------|----------|---------|-------------------|----------------|-----|------|-----|
| 1 | Linear Regression | Raw Price | `get_dummies` | None | All features | None | $33,923,216 | $5,824 | $4,071 ⭐ |
| 2 | Linear Regression | Log Price | `get_dummies` | None | All features | None | $41,450,429 | $6,438 | $4,334 |
| 3 | Linear Regression Pipeline | Log Price | `OneHotEncoder` | None | All features | None | $96,069,073 | $9,801 | $7,044 ❌ |
| 4 | Linear Regression + Polynomial | Log Price | `OneHotEncoder` | None | All features (poly degree=2 on year/odometer) | None | $83,082,080 | $9,115 | $6,448 |
| 5 | Ridge + SFS | Log Price | `OneHotEncoder` | `StandardScaler` | Top 50 (SFS forward) | L2 (default α=1.0) | $44,427,971 | $6,665 | $4,550 ✓ |
| 6 | Lasso + SFS | Log Price | `OneHotEncoder` | `StandardScaler` | Top 50 (SFS forward) | L1 (α=0.01) | $50,899,392 | $7,134 | $4,921 |



In [40]:


numerical_cols = ['year', 'odometer']
categorical_cols = ['manufacturer',  'cylinders', 'transmission', 'fuel', 'drive', 'state', 'condition', 'title_status']

preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numerical_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore', drop='first', sparse_output=False), categorical_cols)
]).set_output(transform='pandas')


pipeline_lr = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LinearRegression( fit_intercept=False) )
]).set_output(transform='pandas')

kf = KFold(n_splits=5, shuffle=True, random_state=42)

cv_results = cross_validate(
    pipeline_lr, X_train, y_train_log,
    cv=kf,
    scoring=['r2', 'neg_mean_absolute_error', 'neg_root_mean_squared_error'],
    return_train_score=True
)

print(f"Train R²  : {cv_results['train_r2'].mean():.4f}")
print(f"Val R²    : {cv_results['test_r2'].mean():.4f}")
print(f"Val MAE   : ${-cv_results['test_neg_mean_absolute_error'].mean():,.2f}")
print(f"Val RMSE  : ${-cv_results['test_neg_root_mean_squared_error'].mean():,.2f}")

pipeline_lr.fit(X_train, y_train_log)

y_pred_log    = pipeline_lr.predict(X_test)
y_pred_actual = np.expm1(y_pred_log)
y_test_actual = np.expm1(y_test_log)

mse  = mean_squared_error(y_test_actual, y_pred_actual)
rmse = np.sqrt(mse)
mae  = np.mean(np.abs(y_test_actual - y_pred_actual))
r2   = pipeline_lr.score(X_test, y_test_log)

print(f"Test R²   : {r2:.4f}")
print(f"Test MAE  : ${mae:,.2f}")
print(f"Test RMSE : ${rmse:,.2f}")

Train R²  : 0.3105
Val R²    : 0.2930
Val MAE   : $0.48
Val RMSE  : $0.92
Test R²   : 0.2926
Test MAE  : $4,518.27
Test RMSE : $6,614.75


In [42]:
feature_names = pipeline_lr.named_steps['preprocessor'].get_feature_names_out()
coefficients = pipeline_lr.named_steps['model'].coef_

coef_df = pd.DataFrame({
    'feature'    : feature_names,
    'coefficient': coefficients
}).sort_values('coefficient', key=abs, ascending=False)

print(coef_df.to_string())

for _, row in coef_df[coef_df['feature'].str.startswith('num__')].iterrows():
    pct_change = (np.exp(row['coefficient']) - 1) * 100
    print(f"{row['feature']}: {pct_change:+.1f}% per std dev")


top_n = coef_df.head(20).copy()
top_n['pct_impact'] = (np.exp(top_n['coefficient']) - 1) * 100

fig = px.bar(
    top_n,
    x='pct_impact',
    y='feature',
    orientation='h',
    title='Top 20 Features by Price Impact (%)',
    labels={'pct_impact': '% Price Change vs Baseline', 'feature': ''},
    color='pct_impact',
    color_continuous_scale='RdYlGn'
)
fig.update_layout(yaxis={'categoryorder': 'total ascending'})
fig.write_html('../images/feature_importance_from_coeffs.html')
fig.show()

                                            feature  coefficient
37                          cat__manufacturer_LOTUS    12.137506
48           cat__manufacturer_PIERCE MANUFACTURING    11.631508
27              cat__manufacturer_INDIAN MOTORCYCLE    11.311091
64                      cat__manufacturer_WORKHORSE    11.023345
49                       cat__manufacturer_PLYMOUTH    10.925600
55                 cat__manufacturer_STERLING TRUCK    10.689547
24                         cat__manufacturer_HUMMER    10.542481
38                           cat__manufacturer_MACK    10.515358
19                            cat__manufacturer_GEO    10.507085
3                      cat__manufacturer_AM GENERAL    10.396230
96                        cat__manufacturer_porsche    10.380330
36                        cat__manufacturer_LINCOLN    10.377700
2                      cat__manufacturer_ALFA ROMEO    10.359563
47                      cat__manufacturer_PETERBILT    10.324249
8                        

### Deployment

Now that we've settled on our models and findings, it is time to deliver the information to the client.  You should organize your work as a basic report that details your primary findings.  Keep in mind that your audience is a group of used car dealers interested in fine-tuning their inventory.